# Diagonal integration: [multiple RNA, multiple ATAC]

Diagonal integration combines RNA and ATAC measured in different cells, with no pairing between them. This tutorial is for data in several batches: three RNA batches and three ATAC batches. It runs iNMF and online_iNMF on `D37mini`, the RNA and ATAC of 3,000 cells of the benchmark dataset `D37` with the pairing left out, then on data in your own format.

iNMF and online_iNMF read ATAC as gene-activity scores, made beforehand with a tool such as Signac or ArchR.

Methods run on Linux. On macOS or Windows, [open this notebook in Colab](https://colab.research.google.com/github/DSichang/scMultiBench/blob/main/notebooks/tutorial_diagonal_multi.ipynb): sections 1 to 4 take about 6 minutes there.

## 1. Install

This cell installs `multibench-sc`.

In [ ]:
import numpy, pandas
pins = f"numpy=={numpy.__version__} pandas=={pandas.__version__}"
%pip install -q "multibench-sc>=0.5.1" {pins}

## 2. Download the data and the environments

`mtb.data.fetch` downloads `D37mini` (12 MB) once. `mtb.env.install` downloads the environments for iNMF and online_iNMF. The download is 0.9 GB.

In [ ]:
import multibench as mtb
import pandas as pd

pd.set_option("display.max_columns", None)   # show every column of the tables

mtb.data.fetch("D37mini")
METHODS = ["iNMF", "online_iNMF"]
envs = mtb.env.install(METHODS)

## 3. Run the methods

`run` runs each method on `D37mini` and [scores its output](https://dsichang.github.io/scMultiBench/tutorials/evaluate/#what-each-metric-measures). The clustering metrics score how well the embedding's clusters and neighbourhoods match the cell types, and how separate the clusters are. The batch metrics measure how well the batches mix. The classification scores measure how well the cell types learned on one batch are predicted in the other batches. Higher is better for every metric.

<details>
<summary>Details</summary>

`params={"Method": {"key": value}}` sets a method's parameters, and `mtb.params_for` lists them. Many methods take none.

`mtb.load_run("out/D37mini")` reloads these results later without running anything.

</details>

In [ ]:
res = mtb.run("D37mini", "diagonal", methods=METHODS, out_dir="out/D37mini")
res.summary

## 4. Plot

`res.plot()` draws the scores as a bubble table. Circle size ranks the rows within each column and colour shows the value scaled per column, so a small gap can look large: `res.summary` has the values.

<details>
<summary>Details</summary>

Metrics are grouped by family: blue for clustering, green for batch correction, pink for classification. Each family starts with an `Overall` bar: each row's mean rank over the family's metrics, scaled so that the lowest gets the shortest bar.

A column with fewer than two different values is drawn grey. A figure of one method is grey everywhere.

</details>

In [ ]:
res.plot()

## 5. Your own data

Your data needs raw RNA counts and ATAC as one `.h5ad` file per batch, with a cell type for each cell. online_iNMF needs about 5,000 cells in total.

Starting from something else? See the recipes for [labels in a CSV file](https://dsichang.github.io/scMultiBench/tutorials/run/#recipe-labels-csv) and [no labels](https://dsichang.github.io/scMultiBench/tutorials/run/#recipe-no-labels).

<details>
<summary>Details: export</summary>

The names in quotes say where your file holds each part. `"X"` is the main matrix, `"obsm:atac"` the matrix `obsm["atac"]`, `"layer:counts"` the layer `layers["counts"]`, and `"obs:celltype"` the column `obs["celltype"]`. A wrong name after `obsm:`, `layer:` or `obs:` raises an error that lists the names the file has.

The RNA batches and the ATAC batches are numbered separately. RNA batch 1 and ATAC batch 1 need not come from the same sample. scJoint, GLUE, iNMF and online_iNMF read exactly three RNA and three ATAC batches. For another number, such as two donors, export `rna` and `atac` with every cell, without `batch_index=`, and pass the batches to `run` as `batch=`, one per RNA and ATAC cell, as in [another number of donors](https://dsichang.github.io/scMultiBench/tutorials/run/#recipe-donors).

For ATAC as a peak matrix, pass `atac_kind="peak"`. GLUE reads the peak matrix.

The folder stores each matrix dense, so drop the genes or peaks you do not need first.

</details>

So that the notebook runs as it is, the next cell writes six small example files, `my_rna1.h5ad` to `my_rna3.h5ad` and `my_atac1.h5ad` to `my_atac3.h5ad`, from the first 900 RNA profiles and the first 900 ATAC profiles of each batch of `D37mini`.

In [ ]:
mtb.data.write_example("D37mini")

Read your files. For your own data, change every file name marked `# <-`. On Colab, upload the files first: click the folder icon on the left, then the upload icon.

In [ ]:
import anndata as ad

rna1 = ad.read_h5ad("my_rna1.h5ad")     # <- RNA batch 1: your file
rna2 = ad.read_h5ad("my_rna2.h5ad")     # <- RNA batch 2: your file
rna3 = ad.read_h5ad("my_rna3.h5ad")     # <- RNA batch 3: your file
atac1 = ad.read_h5ad("my_atac1.h5ad")   # <- ATAC batch 1: your file
atac2 = ad.read_h5ad("my_atac2.h5ad")   # <- ATAC batch 2: your file
atac3 = ad.read_h5ad("my_atac3.h5ad")   # <- ATAC batch 3: your file
print(rna1)
print(atac1)

Write the dataset folder. Change each name in quotes marked `# <-`. The printout above lists the names in two of your files. Write a name listed after `obs:` with that prefix, as `"obs:celltype"`.

In [ ]:
NAME = "MYDIAG_BATCHES"        # <- a new name for your own data
rna = [rna1, rna2, rna3]       # <- your RNA batches
atac = [atac1, atac2, atac3]   # <- your ATAC batches
for b in range(len(rna)):      # one call per batch
    mtb.io.export_dataset(rna[b], "mydata/" + NAME,
                          rna="X",                    # <- the RNA counts:
                                                      #    "X" or "layer:counts"
                          atac=atac[b],               # the ATAC batch
                          atac_kind="gene_activity",  # <- or "peak"
                          labels="obs:celltype",      # <- the cell-type column
                          category="diagonal", batch_index=b + 1,
                          overwrite=True)
mtb.scan("mydata/" + NAME, "diagonal", methods=METHODS)

In [ ]:
mine = mtb.run("mydata/" + NAME, "diagonal", methods=METHODS,
               out_dir="out/" + NAME)
mine.summary

In [ ]:
mine.plot()

## 6. Every method of this task

4 methods run this task. [This notebook](https://colab.research.google.com/github/DSichang/scMultiBench/blob/main/notebooks/tutorial_diagonal_multi_all.ipynb) runs them all. It downloads 3 envs, 4.8 GB on a computer without a GPU and 8.3 GB on one with a GPU.

## Troubleshooting

`res.summary[res.summary.status != "scored"]` lists each method that failed or was skipped. Its `reason` column ends with the method's error output.

<details>
<summary>Details</summary>

| symptom | fix |
|---|---|
| a method fails on a small dataset | `mtb.scan` names what the method needs from the data's size in its `caveat` column |
| a method times out | raise `timeout=` in `run` |
| batch metrics use the wrong batches | `res.rescore(batch=my_vector)` scores again without running the methods |

</details>

## Next steps

- The guides: [run](https://dsichang.github.io/scMultiBench/tutorials/run/), [evaluate](https://dsichang.github.io/scMultiBench/tutorials/evaluate/), [plot](https://dsichang.github.io/scMultiBench/tutorials/plot/) and [discover methods](https://dsichang.github.io/scMultiBench/tutorials/discover/).
- The [interactive explorer](https://shiny.maths.usyd.edu.au/scMultiBench/) has the full benchmark's rankings, with no install needed.